In [4]:
# Import statements
from oqd_core.interface.analog import (
    AnalogCircuitSSA, Block, Branch, CondBranch, Exit,
    AnalogGate, Evolve, PauliX, PauliZ,
    QuantumDeclaration, ClassicalDeclaration,
    RegisterNonZero, SSAValBool,
    SSADefBool, SSADefMath,
)
from oqd_core.interface.analog.register import ClassicalRef, ClassicalRegister
from oqd_core.compiler.analog.passes.resolve import resolve_analog
from oqd_core.interface.math import MathNum, MathRef, SSAValMath
from oqd_compiler_infrastructure import Post, PrettyPrint

printer = Post(PrettyPrint())

In [5]:
# Single block with Evolve and Exit terminator (no branching)
X = PauliX()
circuit = AnalogCircuitSSA(
    declarations=[QuantumDeclaration(name="q", size=2)],
    blocks=[
        Block(
            label="entry",
            args=[],
            body=[
                Evolve(gate=AnalogGate(hamiltonian=X * 0.5), duration=2.0),
            ],
            terminator=Exit(),
        ),
    ],
)
print(printer(circuit))

AnalogCircuitSSA
  - qreg: list
  - creg: list
  - declarations: list
    - 0: QuantumDeclaration
      - name: str(q)
      - size: int(2)
  - blocks: list
    - 0: Block
      - label: str(entry)
      - args: list
      - body: list
        - 0: Evolve
          - key: str(evolve)
          - duration: float(2.0)
          - gate: AnalogGate
            - hamiltonian: OperatorScalarMul
              - op: PauliX
              - expr: MathNum
                - value: float(0.5)
      - terminator: Exit


In [6]:
# IfElse expressed as blocks: entry branches on condition to then/else, both merge to exit
X, Z = PauliX(), PauliZ()
circuit = AnalogCircuitSSA(
    declarations=[
        QuantumDeclaration(name="q", size=2),
        ClassicalDeclaration(name="c", size=1),
    ],
    blocks=[
        Block(
            label="entry",
            args=[],
            body=[Evolve(gate=AnalogGate(hamiltonian=X * 0.5), duration=1.0)],
            terminator=CondBranch(
                condition=RegisterNonZero(creg=ClassicalRef(name="c")),
                true_target="then",
                true_args=[],
                false_target="else",
                false_args=[],
            ),
        ),
        Block(
            label="then",
            args=[],
            body=[Evolve(gate=AnalogGate(hamiltonian=Z * 0.1), duration=1.0)],
            terminator=Branch(target="merge", args=[]),
        ),
        Block(
            label="else",
            args=[],
            body=[Evolve(gate=AnalogGate(hamiltonian=X * 0.2), duration=1.5)],
            terminator=Branch(target="merge", args=[]),
        ),
        Block(label="merge", args=[], body=[], terminator=Exit()),
    ],
)
print(printer(circuit))

AnalogCircuitSSA
  - qreg: list
  - creg: list
  - declarations: list
    - 0: QuantumDeclaration
      - name: str(q)
      - size: int(2)
    - 1: ClassicalDeclaration
      - name: str(c)
      - size: int(1)
  - blocks: list
    - 0: Block
      - label: str(entry)
      - args: list
      - body: list
        - 0: Evolve
          - key: str(evolve)
          - duration: float(1.0)
          - gate: AnalogGate
            - hamiltonian: OperatorScalarMul
              - op: PauliX
              - expr: MathNum
                - value: float(0.5)
      - terminator: CondBranch
        - condition: RegisterNonZero
          - creg: ClassicalRef
            - name: str(c)
            - index: NoneType(None)
        - true_target: str(then)
        - true_args: list
        - false_target: str(else)
        - false_args: list
    - 1: Block
      - label: str(then)
      - args: list
      - body: list
        - 0: Evolve
          - key: str(evolve)
          - duration: float(1.0)
 

In [7]:
# Block-local SSA definitions: define omega via SSADefMath, use SSAValMath in Evolve
X = PauliX()
circuit = AnalogCircuitSSA(
    declarations=[QuantumDeclaration(name="q", size=2)],
    blocks=[
        Block(
            label="entry",
            args=[],
            body=[
                SSADefMath(name="omega", expr=MathNum(value=0.5)),
                Evolve(
                    gate=AnalogGate(hamiltonian=X * SSAValMath(name="omega")),
                    duration=2.0,
                ),
            ],
            terminator=Exit(),
        ),
    ],
)
print(printer(circuit))

AnalogCircuitSSA
  - qreg: list
  - creg: list
  - declarations: list
    - 0: QuantumDeclaration
      - name: str(q)
      - size: int(2)
  - blocks: list
    - 0: Block
      - label: str(entry)
      - args: list
      - body: list
        - 0: SSADefMath
          - name: str(omega)
          - expr: MathNum
            - value: float(0.5)
        - 1: Evolve
          - key: str(evolve)
          - duration: float(2.0)
          - gate: AnalogGate
            - hamiltonian: OperatorScalarMul
              - op: PauliX
              - expr: SSAValMath
                - name: str(omega)
      - terminator: Exit


In [8]:
# Resolve circuit-level refs (ClassicalRef, MathRef) via resolve_analog
X, Z = PauliX(), PauliZ()
circuit = AnalogCircuitSSA(
    declarations=[
        QuantumDeclaration(name="q", size=2),
        ClassicalDeclaration(name="c", size=1),
        # MathExprDeclaration(name="omega", expr=MathNum(value=0.3)),
    ],
    blocks=[
        Block(
            label="entry",
            args=[],
            body=[
                Evolve(gate=AnalogGate(hamiltonian=X * 0.5), duration=1.0),
            ],
            terminator=CondBranch(
                condition=RegisterNonZero(creg=ClassicalRef(name="c")),
                true_target="then",
                true_args=[],
                false_target="else",
                false_args=[],
            ),
        ),
        Block(
            label="then",
            args=[],
            body=[Evolve(gate=AnalogGate(hamiltonian=Z * 0.1), duration=1.0)],
            terminator=Branch(target="exit", args=[]),
        ),
        Block(
            label="else",
            args=[],
            body=[Evolve(gate=AnalogGate(hamiltonian=X * 0.2), duration=1.5)],
            terminator=Branch(target="exit", args=[]),
        ),
        Block(label="exit", args=[], body=[], terminator=Exit()),
    ],
)
resolved = resolve_analog(circuit)
print(printer(resolved))

AnalogCircuitSSA
  - qreg: list
  - creg: list
  - declarations: list
    - 0: QuantumDeclaration
      - name: str(q)
      - size: int(2)
    - 1: ClassicalDeclaration
      - name: str(c)
      - size: int(1)
  - blocks: list
    - 0: Block
      - label: str(entry)
      - args: list
      - body: list
        - 0: Evolve
          - key: str(evolve)
          - duration: float(1.0)
          - gate: AnalogGate
            - hamiltonian: OperatorScalarMul
              - op: PauliX
              - expr: MathNum
                - value: float(0.5)
      - terminator: CondBranch
        - condition: RegisterNonZero
          - creg: ClassicalRegister
            - id: str(c)
            - reg: list
              - 0: ClassicalBit
                - id: str(c)
                - index: int(0)
        - true_target: str(then)
        - true_args: list
        - false_target: str(else)
        - false_args: list
    - 1: Block
      - label: str(then)
      - args: list
      - body: list

In [9]:
# Block with args: loop receives i, sum from predecessor; simulates block-argument SSA
circuit = AnalogCircuitSSA(
    declarations=[QuantumDeclaration(name="q", size=2)],
    blocks=[
        Block(
            label="entry",
            args=[],
            body=[
                SSADefMath(name="omega", expr=MathNum(value=0.1)),
                Evolve(gate=AnalogGate(hamiltonian=X * 0.5), duration=1.0),
            ],
            terminator=Branch(target="loop", args=[SSAValMath(name="omega"), MathNum(value=0.0)]),
        ),
        Block(
            label="loop",
            args=["omega", "phase"],  # block args from predecessor's Branch
            body=[
                Evolve(
                    gate=AnalogGate(hamiltonian=X * SSAValMath(name="omega")),
                    duration=1.0,
                ),
            ],
            terminator=Exit(),
        ),
    ],
)
print(printer(circuit))

AnalogCircuitSSA
  - qreg: list
  - creg: list
  - declarations: list
    - 0: QuantumDeclaration
      - name: str(q)
      - size: int(2)
  - blocks: list
    - 0: Block
      - label: str(entry)
      - args: list
      - body: list
        - 0: SSADefMath
          - name: str(omega)
          - expr: MathNum
            - value: float(0.1)
        - 1: Evolve
          - key: str(evolve)
          - duration: float(1.0)
          - gate: AnalogGate
            - hamiltonian: OperatorScalarMul
              - op: PauliX
              - expr: MathNum
                - value: float(0.5)
      - terminator: Branch
        - target: str(loop)
        - args: list
          - 0: SSAValMath
            - name: str(omega)
          - 1: MathNum
            - value: float(0.0)
    - 1: Block
      - label: str(loop)
      - args: list
        - 0: str(omega)
        - 1: str(phase)
      - body: list
        - 0: Evolve
          - key: str(evolve)
          - duration: float(1.0)
        